# Create DSL search queries

In [ ]:
import json
from datetime import datetime
from pathlib import Path
from typing import Any

from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

In [ ]:
docs = Path("..") / "docs"
multi_source_dsl_path = (
    docs / "coresignal" / "Employee APIs" / "Multi-source Employee API.json"
)

with open(multi_source_dsl_path, "r") as file:
    multi_source_dsl_json = file.read()

In [ ]:
# llm = ChatGoogleGenerativeAI(
#     model="gemini-3.7-flash",
#     temperature=1.0,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
# )
llm = ChatAnthropic(model="claude-sonnet-5")  # type: ignore

In [ ]:
system_prompt = f"""
Generate a DSL search query for the job description

DSL Description:
```json
{multi_source_dsl_json}
```
""".strip()

In [ ]:
openenings = Path("..") / "openings"
job_description_path = openenings / "frontend-engineer-berlin.md"

with open(job_description_path, "r") as file:
    job_description = file.read()

In [ ]:
# 1. Define your desired output structure using Pydantic
class DSLQuery(BaseModel):
    query: dict[str, Any] = Field(description="The DSL search query")

In [ ]:
messages = [
    ("system", system_prompt),
    ("human", job_description),
]

dsl_gen = llm.with_structured_output(DSLQuery, method="function_calling")
raw = dsl_gen.invoke(messages)
result = DSLQuery.model_validate(raw)

In [ ]:
output_dir = Path("..") / "spi" / "search_query"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + ".json"

with open(output_dir / output_file, "w") as file:
    json.dump(result.query, file, indent=2)

In [ ]:
print(json.dumps(result.query, indent=2))